In [ ]:
#Import the necessary libraries
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
#Object detecting class
class ObjectDetector3D:
    def __init__(self, model_path, camera_index=0):
        """
        Initialize the object detector with YOLOv8 model

        Args:
            model_path: Path to your trained YOLOv8 model (.pt file)
            camera_index: Camera device index (default 0)
        """
        self.model = YOLO(model_path)
        self.camera_index = camera_index
        self.cap = None

        # Camera intrinsic parameters (calibrate these for your specific camera)
        self.focal_length = 500  # pixels (approximate, needs calibration)
        self.frame_width = None
        self.frame_height = None
        self.cx = None  # optical center x
        self.cy = None  # optical center y

        # Known object sizes (in cm) - adjust based on your animals
        # Format: {class_name: average_size_in_cm}
        self.object_sizes = {
            'cat': 25,
            'dog': 40,
            'bird': 15,
            # Add your animal classes here
        }

    def open_camera(self):
        """
        Open the camera and initialize camera parameters

        Returns:
            bool: True if camera opened successfully, False otherwise
        """
        self.cap = cv2.VideoCapture(self.camera_index)

        if not self.cap.isOpened():
            print(f"Error: Could not open camera {self.camera_index}")
            return False

        # Get camera properties
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.cx = self.frame_width / 2  # optical center x
        self.cy = self.frame_height / 2  # optical center y

        print(f"Camera opened successfully: {self.frame_width}x{self.frame_height}")
        return True

    def close_camera(self):
        """
        Release the camera resource
        """
        if self.cap is not None:
            self.cap.release()
            print("Camera closed")
        cv2.destroyAllWindows()

    def estimate_distance(self, bbox_width, bbox_height, class_name):
        """
        Estimate distance (Z coordinate) using object size and focal length

        Args:
            bbox_width: Width of bounding box in pixels
            bbox_height: Height of bounding box in pixels
            class_name: Detected object class

        Returns:
            distance in cm
        """
        if class_name not in self.object_sizes:
            return None

        # Use the larger dimension for better accuracy
        pixel_size = max(bbox_width, bbox_height)
        real_size = self.object_sizes[class_name]

        # Distance = (Real Size × Focal Length) / Pixel Size
        distance = (real_size * self.focal_length) / pixel_size
        return distance

    def pixel_to_3d(self, x_pixel, y_pixel, z_distance):
        """
        Convert pixel coordinates to 3D camera coordinates

        Args:
            x_pixel: X coordinate in image (pixels)
            y_pixel: Y coordinate in image (pixels)
            z_distance: Distance from camera (cm)

        Returns:
            (x, y, z) in cm relative to camera
        """
        # Convert pixel coordinates to camera coordinates
        x_cam = (x_pixel - self.cx) * z_distance / self.focal_length
        y_cam = (y_pixel - self.cy) * z_distance / self.focal_length
        z_cam = z_distance

        return x_cam, y_cam, z_cam

    def detect_and_localize(self, frame):
        """
        Run detection and compute 3D coordinates for all detected objects

        Args:
            frame: Input image frame

        Returns:
            List of detections with 3D coordinates
        """
        results = self.model(frame, verbose=False)
        detections = []

        for result in results:
            boxes = result.boxes
            for box in boxes:
                # Extract bounding box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                confidence = box.conf[0].cpu().numpy()
                class_id = int(box.cls[0].cpu().numpy())
                class_name = self.model.names[class_id]

                # Calculate center point and dimensions
                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2
                bbox_width = x2 - x1
                bbox_height = y2 - y1

                # Estimate distance (Z coordinate)
                z_distance = self.estimate_distance(bbox_width, bbox_height, class_name)

                if z_distance:
                    # Convert to 3D coordinates
                    x_3d, y_3d, z_3d = self.pixel_to_3d(center_x, center_y, z_distance)

                    detections.append({
                        'class': class_name,
                        'confidence': float(confidence),
                        'bbox': (int(x1), int(y1), int(x2), int(y2)),
                        'center_2d': (int(center_x), int(center_y)),
                        'coordinates_3d': (x_3d, y_3d, z_3d)
                    })

        return detections

    def draw_detections(self, frame, detections):
        """
        Draw bounding boxes and 3D coordinates on frame

        Args:
            frame: Input image frame
            detections: List of detection dictionaries

        Returns:
            Annotated frame
        """
        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            center_x, center_y = det['center_2d']
            x_3d, y_3d, z_3d = det['coordinates_3d']

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Draw center point
            cv2.circle(frame, (center_x, center_y), 5, (0, 0, 255), -1)

            # Display class and confidence
            label = f"{det['class']}: {det['confidence']:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Display 3D coordinates
            coord_text = f"X:{x_3d:.1f} Y:{y_3d:.1f} Z:{z_3d:.1f}cm"
            cv2.putText(frame, coord_text, (x1, y2 + 20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        return frame

    def run(self):
        """
        Main loop for real-time detection
        """
        # Open camera first
        if not self.open_camera():
            return

        print("Starting object detection... Press 'q' to quit")

        try:
            while True:
                ret, frame = self.cap.read()
                if not ret:
                    print("Failed to grab frame")
                    break

                # Detect objects and compute 3D coordinates
                detections = self.detect_and_localize(frame)

                # Print detections to console
                if detections:
                    for det in detections:
                        x, y, z = det['coordinates_3d']
                        print(f"{det['class']}: X={x:.2f}cm, Y={y:.2f}cm, Z={z:.2f}cm")

                # Exit on 'q' key
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
        finally:
            # Always close camera when done
            self.close_camera()